In [1]:
import os
import torch
import pandas as pd
import numpy as np
import random
import json
from numpy.lib.format import open_memmap
from torch.nn import functional as F
from accelerate import Accelerator, find_executable_batch_size
from transformers import AutoTokenizer, AutoModelForCausalLM

In [2]:
device = "cpu"
if torch.cuda.is_available():
    device = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    device = "mps"
print(f"Using device: {device}")

NP_FILE = "noun_phrases.csv"
LOGPROBS_DIR = "logprobs"
df = pd.read_csv(NP_FILE)

N_SAMPLES = 100

SEED = 1568
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

accelerator = Accelerator()

Using device: mps


In [ ]:
model_paths = {
    'lambda0p0': 'weijiexu97/nanogpt-noisy-lambda0p0-wt103-bs512',
    'lambda0p001': 'weijiexu97/nanogpt-noisy-lambda0p001-wt103-bs512',
    'lambda0p01': 'weijiexu97/nanogpt-noisy-lambda0p01-wt103-bs512',
    'lambda0p1': 'weijiexu97/nanogpt-noisy-lambda0p1-wt103-bs512',
    'lambda1p0': 'weijiexu97/nanogpt-noisy-lambda1p0-wt103-bs512'
}
model_names = ['lambda0p0', 'lambda0p001', 'lambda0p01', 'lambda0p1', 'lambda1p0']

tokenizer = AutoTokenizer.from_pretrained("weijiexu97/nanogpt-noisy-lambda0p0-wt103-bs512", trust_remote_code=True)
if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
print(f"pad token id: {tokenizer.pad_token_id}")

def load_model(model_path, device=device):
    model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True)
    model.to(device)
    model.eval()
    return model

pad token id: 50256


In [4]:
num_sent = len(df)
print(num_sent)

df["logits_idx"] = df.index.astype(np.int64)
sentences = df["np_text"].astype(str).tolist()
enc = tokenizer(
        sentences,
        return_offsets_mapping=True,
        padding=True,
        truncation=False,
        add_special_tokens=False,
    )
data = tensor = torch.tensor(enc['input_ids'])

context_len = data.shape[1]
print(data.shape)
data

1000
torch.Size([1000, 8])


tensor([[  464,  8674,   287,  ...,  3221, 50256, 50256],
        [  464, 14549,  1088,  ...,  3221, 50256, 50256],
        [  464, 18727,   739,  ...,  3221, 50256, 50256],
        ...,
        [  464,  2497,   287,  ...,  3221, 50256, 50256],
        [  464,  3159,  2641,  ...,  3221, 50256, 50256],
        [  464, 18316,   287,  ...,  3221, 50256, 50256]])

In [5]:
def select_last_positions(batch):
    valid_len = (batch != tokenizer.pad_token_id).sum(dim=1)
    tgt_idx = valid_len - 1  # the target token position
    keep_mask = valid_len >= 2
    return valid_len, tgt_idx, keep_mask

def gather_rowwise_logits(logits, row_pos_idx):
    B = logits.size(0)
    array = torch.arange(B, device=logits.device)
    return logits[array, row_pos_idx, :]

In [6]:
def get_logprobs_next_token(df, data, model, model_name, n_layers, n_heads):

    rowidx = df["logits_idx"].to_numpy()

    @find_executable_batch_size(starting_batch_size=128)
    def inference_loop(batch_size):
        accelerator.free_memory()
        print(f"batch_size={batch_size}")

        dl = torch.utils.data.DataLoader(data, batch_size=batch_size, shuffle=False)

        logprobs_ls = []

        cursor = 0

        with torch.inference_mode():
            for batch in dl:
                batch_cpu = batch
                valid_len, tgt_idx, keep_mask = select_last_positions(batch_cpu)
                B = batch_cpu.size(0)
                batch = batch.to(device)
                tgt_idx = tgt_idx.to(device)
                global_idx = torch.from_numpy(rowidx[cursor:cursor + B])

                if model_name in ['base_gpt2']:
                    output = model(batch)
                    logits = output.logits
                    logits_tgt = gather_rowwise_logits(logits, tgt_idx)
                    logprobs_tgt = torch.log_softmax(logits_tgt, dim=-1)
                    logprobs_ls.append(logprobs_tgt.detach().cpu())
                else:
                    # ----- noisy models: average over N_SAMPLES -----
                    logprobs_samples = []
                    for _ in range(N_SAMPLES):
                        output = model(batch)
                        logits = output.logits
                        logits_tgt = gather_rowwise_logits(logits, tgt_idx)
                        logprobs_tgt = torch.log_softmax(logits_tgt, dim=-1)
                        logprobs_samples.append(logprobs_tgt.detach().cpu())
                    mean_logprobs = torch.stack(logprobs_samples, dim=0).mean(dim=0)
                    logprobs_ls.append(mean_logprobs)
                
            logprobs_stacked = torch.vstack(logprobs_ls)
            print(logprobs_stacked.shape)
        
        return logprobs_stacked
    
    logprobs_stacked = inference_loop()

    return logprobs_stacked.numpy()

In [ ]:
os.makedirs(LOGPROBS_DIR, exist_ok=True)

for model_name in model_names:
    print(f"\n=== START {model_name} ===")
    model = load_model(model_paths[model_name], device=device)

    cfg = model.config
    n_layers = int(cfg.n_layer)
    n_heads = int(cfg.n_head)

    logits = get_logprobs_next_token(df, data, model, model_name, n_layers, n_heads)
    np.save(f'{LOGPROBS_DIR}/logprobs_{model_name}.npy', logits)
    
    print("\nWrote logprobs:", f'{LOGPROBS_DIR}/logprobs_{model_name}.npy')